## Packages

In [ ]:
# Standard library
import os
import re
import time
import glob
import math

# Science
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.io import savemat
from scipy import signal
from scipy.stats import ttest_rel

# Plotting
import matplotlib.pyplot as plt
import matplotlib as mpl

# Scikit learn
from sklearn.cross_decomposition import CCA
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC

## Variables and small file import

In [ ]:
# 9 Channels chosen to fit active occipital/parietal region: 
# Pz, PO5, PO3, POz, PO4, PO6, O1, Oz, O2
channel_labels = ['Pz', 'PO5', 'PO3', 'POz', 'PO4', 'PO6', 'O1', 'Oz', 'O2']
channels = [47, 53, 54, 55, 56, 57, 60, 61, 62]

label_to_idx = {lab: i for i, lab in enumerate(channel_labels)}
fs = 250 
sub_n = 35

#paths
base_path = r'\data'
label_path = base_path + r'\Freq_Phase.mat'
meta_path = base_path + r'\subject_info_35_dataSets.txt'

# load labels from trials
labels_fr = sio.loadmat(label_path)['freqs']
labels_ph = sio.loadmat(label_path)['phases']

# load meta data
metadata = open(meta_path)

## Batch Import and channel selection

In [ ]:
# Iterate over the data of 35 subjects
subdata_channels = []

for sub in range(sub_n):
    subdata_path = base_path + r'\S' + str(sub + 1) + r'.mat'
 
    # subdata shape: [n_channels, n_samples, n_trials, n_trial_blocks]
    subdata = sio.loadmat(subdata_path)['data']

    # extract the channels of interest
    sub_ch = subdata[channels, :, :, :] 

    subdata_channels.append(sub_ch)

    # Print the subject number and the shape of EEG matrices for progress
    print('Subject ' + str(sub + 1))
    print(sub_ch.shape)

## Pre-processing (filtering) module

In [ ]:
nB, nA = signal.iircomb(50, 35, ftype='notch', fs=fs)

# Iterate over the data of 35 subjects
filtered_data = []

for sub in range(sub_n):
    subdata_raw = subdata_channels[sub]

    n_data = signal.filtfilt(nB, nA, subdata_raw, axis = 1, padtype='odd', padlen=3*(max(len(nB),len(nA))-1))
    filtered_data.append(n_data)

    # Print the subject number and the shape of EEG matrices for progress
    print('Subject ' + str(sub + 1) + ' filtered')

## Feature extraction

In [ ]:
def cca_reference(list_freqs, fs, n_smpls, n_harms=3):
    """
    y_ref[k] = reference sinusoids
    Output: (K, 2H, T)
    """
    f = np.asarray(list_freqs, dtype=float).reshape(-1)               # (K,)
    t = np.arange(n_smpls, dtype=float) / fs                          # (T,)
    h = np.arange(1, n_harms + 1, dtype=float)                        # (H,)

    ang = 2.0 * np.pi * f[:, None, None] * h[None, :, None] * t[None, None, :]  # (K,H,T)

    y_ref = np.empty((f.size, 2 * n_harms, n_smpls), dtype=float)
    y_ref[:, 0::2, :] = np.sin(ang)
    y_ref[:, 1::2, :] = np.cos(ang)
    return y_ref

def filterbank_sos(fs, num_fbs=7, f_high=90.0, step_hz=8.0,
                           low_trans_hz=2.0, high_trans_hz=10.0,
                           gpass=3, gstop=40, rp=0.5):
    """
    Chen 2015: subband k has passband [step_hz*k, f_high]
    with stopband [step_hz*k - low_trans_hz, f_high + high_trans_hz].
    Returns a list of SOS filters.
    """
    nyq = fs / 2.0
    f_high = min(float(f_high), 0.98 * nyq)
    f_stop_high = min(f_high + float(high_trans_hz), 0.98 * nyq)

    sos_list = []

    for k in range(1, num_fbs + 1):
        wp_low = step_hz * k
        ws_low = max(0.1, wp_low - low_trans_hz)

        # Keep within valid range
        wp_low = min(wp_low, 0.98 * nyq)
        ws_low = min(ws_low, 0.98 * nyq)

        N, Wn = signal.cheb1ord([wp_low, f_high], [ws_low, f_stop_high],
                                gpass=gpass, gstop=gstop, fs=fs)
        sos = signal.cheby1(N, rp=rp, Wn=Wn, btype="bandpass", fs=fs, output="sos")
        sos_list.append(sos)

    return sos_list

def fbcca_scores(epoch, y_ref, sos_list, fb_a=1.25, fb_b=0.25):
    """
    epoch: (n_ch, T)
    y_ref: (K, 2H, T)
    returns: rho (K,)
    """
    X0 = np.asarray(epoch, dtype=float)
    K = y_ref.shape[0]
    num_fbs = len(sos_list)

    T = X0.shape[1]
    fb_coefs = (np.arange(1, num_fbs + 1, dtype=float) ** (-fb_a)) + fb_b
    r2 = np.zeros((num_fbs, K), dtype=float)

    for fb_i, sos in enumerate(sos_list):
        try:
            Xf = signal.sosfiltfilt(sos, X0, axis=1).T   # (T, n_ch)
        except ValueError:
            safe_padlen = max(0, T - 1)
            Xf = signal.sosfiltfilt(sos, X0, axis=1, padlen=safe_padlen).T

        for k in range(K):
            Y = y_ref[k].T  # (T, 2H)
            cca = CCA(n_components=1, max_iter=1000)
            Xc, Yc = cca.fit_transform(Xf, Y)
            rr = np.corrcoef(Xc[:, 0], Yc[:, 0])[0, 1]
            r2[fb_i, k] = 0.0 if not np.isfinite(rr) else rr * rr

    rho = fb_coefs @ r2
    return rho


def build_epoch_length_indices(fs, t_cue=0.5, t_latency=0.14, t_gaze=0.5):
    start = int(round((t_cue + t_latency) * fs))
    stop = start + int(round(t_gaze * fs))
    return np.arange(start, stop, dtype=int)


def fbcca_features_per_subject(sub_data, epoch_idx, y_ref, sos_list, ch_label=None):
    """
    Extract FBCCA score vectors for one subject.

    Inputs
      sub_data: (n_ch, n_samples, n_trials, n_blocks)
      epoch_idx: sample indices for the time window
      y_ref: (K, 2*n_harms, T)
      sos_list: list of SOS filters
      ch_label:
        None        -> use all loaded channels
        string      -> use one channel by label

    Output
      feats: (n_trials, n_blocks, K)
    """
    if ch_label is None:
        x = sub_data
    else:
        if ch_label not in label_to_idx:
            raise ValueError(f"Unknown channel label: {ch_label}")
        ch_idx = label_to_idx[ch_label]
        x = sub_data[[ch_idx], :, :, :]   # keep channel dimension

    _, _, n_trials, n_blocks = x.shape
    K = y_ref.shape[0]

    feats = np.zeros((n_trials, n_blocks, K), dtype=float)

    for tr in range(n_trials):
        for bl in range(n_blocks):
            epoch = x[:, epoch_idx, tr, bl]
            feats[tr, bl] = fbcca_scores(epoch, y_ref, sos_list)

    return feats

def phase_estimation(eeg_segment, target_frequencies, predicted_idx):
    fs = 250

    if isinstance(predicted_idx, (int, np.integer)):
        predicted_idx = np.array([predicted_idx])
        return_scalar = True
    else:
        predicted_idx = np.array(predicted_idx)
        return_scalar = False

    n_fft = 10000
    f_data = np.fft.fft(eeg_segment, n=n_fft)
    f_data_shifted = np.squeeze(np.fft.fftshift(f_data))
    f_axis = np.linspace(-fs/2, fs/2, n_fft, endpoint=False)
    phases = np.zeros(len(predicted_idx), dtype=np.float64)

    # Process each requested frequency index
    for i, freq_idx in enumerate(predicted_idx):
        target_freq = target_frequencies[freq_idx]
        freq_mask = (f_axis > target_freq - 0.05) & (f_axis < target_freq + 0.05)
        f_data_in_band = f_data_shifted[freq_mask]

        # Find index of maximum magnitude
        max_idx = np.argmax(np.abs(f_data_in_band))
        freq_bin_indices = np.where(freq_mask)[0]

        # Get phase at the peak
        phases[i] = np.angle(f_data_shifted[freq_bin_indices[max_idx]])

    # Return scalar if input was scalar
    if return_scalar:
        return phases[0]
    return phases


def fbcca_phase_features_per_subject(sub_data, epoch_idx, y_ref, sos_list,
                                    ch_label, channel_labels, target_freqs, fbcca_preds, fs):
    """
    sub_data: (n_ch, n_samples, n_trials, n_blocks) for one subject
    Returns: feats (n_trials, n_blocks, 40 + 2)
    """
    label_to_idx = {lab: i for i, lab in enumerate(channel_labels)}
    K = y_ref.shape[0]
    n_pred = int(fbcca_preds)

    # Decide which channels to use
    if ch_label is None:
        x = sub_data                      
        phase_ch_in_x = 0                 
    else:
        if ch_label not in label_to_idx:
            raise ValueError(f"Unknown channel label: {ch_label}")
        ch_idx = label_to_idx[ch_label]
        x = sub_data[[ch_idx], :, :, :]   
        phase_ch_in_x = 0                 

    _, _, n_trials, n_blocks = x.shape
    feats = np.zeros((n_trials, n_blocks, K + 2 * n_pred), dtype=float)

    for tr in range(n_trials):
        for bl in range(n_blocks):
            epoch = x[:, epoch_idx, tr, bl]                 # (n_ch_used, T)

            # FBCCA scores
            rho = fbcca_scores(epoch, y_ref, sos_list)      # (K,)
            feats[tr, bl, :K] = rho

            pred_idx = np.argsort(rho)[::-1][:n_pred]

            eeg_1ch = epoch[phase_ch_in_x, :]               # (T,)
            ang = phase_estimation(eeg_1ch, target_freqs, pred_idx)  # (n_pred,)

            feats[tr, bl, K:K + 2 * n_pred:2] = np.cos(ang)
            feats[tr, bl, K + 1:K + 2 * n_pred:2] = np.sin(ang)

    return feats


### Feature extraction for both FBCCA and phase

In [ ]:
# Parameters
num_harms = 5 # 2015 FBCCA Chen supports this for accuracy
num_fbs = 7 #2015 FBCCA Chen supports this for accuracy
f_high = 90.0 #2015 FBCCA Chen supports this for the optimal bound for harmonics

t_cue = 0.5 # Value given by the Benchmark dataset
t_latency = 0.14 # Benchmark dataset supports this value for average gaze latency across the dataset
t_w = 2 #2015 FBCCA Chen supports this for accuracy and ITR

# Target frequencies for phase
target_freqs = np.asarray(labels_fr).reshape(-1)

# Indices for the epoch inside each 6 s trial
epoch_idx = build_epoch_length_indices(
    fs=fs,
    t_cue=t_cue,
    t_latency=t_latency,
    t_gaze=t_w
)

# Stimulation freqs
list_freqs = np.asarray(labels_fr).reshape(-1)

# CCA reference frequencies from the known stimulation frequencies
y_ref = cca_reference(
    list_freqs=list_freqs,
    fs=fs,
    n_smpls=len(epoch_idx),
    n_harms=num_harms
)

# Filterbank with number of filters
sos_list = filterbank_sos(
    fs=fs,
    num_fbs=num_fbs,
    f_high=f_high
)

information = np.zeros((len(channel_labels),2))
print(information.shape)

fbcca_preds = 3

all_feats_by_channel = []   # shape (sub_n, 40, 6, n_features)

for i, chosen_channel in enumerate(channel_labels):
    fbcca_phase_feats_ch = []
    print(chosen_channel)

    for sub in range(sub_n):
        fbcca_phase_feats_sub = fbcca_phase_features_per_subject(
            sub_data=filtered_data[sub],
            epoch_idx=epoch_idx,
            y_ref=y_ref,
            sos_list=sos_list,
            ch_label=chosen_channel,
            channel_labels=channel_labels,
            target_freqs=target_freqs,
            fbcca_preds = fbcca_preds,
            fs=fs
        )
        fbcca_phase_feats_ch.append(fbcca_phase_feats_sub)
        print("Subject", sub + 1, "channel", chosen_channel, "done")

    fbcca_feats_ch = np.stack(fbcca_phase_feats_ch, axis=0)
    print("FBCCA features shape:", fbcca_feats_ch.shape)

    all_feats_by_channel.append(fbcca_feats_ch.astype(np.float32))

    # fbcca_feats_list element shape: (40, 6, 40)
    n_subjects, n_trials, n_blocks, n_targets = fbcca_feats_ch.shape

    # Timing for ITR
    tw = t_w
    t_break = 0.5
    t_latency = 0.14
    t_comp = 0.0

    total_t = tw + t_break + t_latency + t_comp
    N = n_trials

    # LOSO evaluation
    cm_total = np.zeros((n_trials, n_trials), dtype=int)
    acc_folds = []
    itr_folds = []

    for test_sub in range(sub_n):
        train_subs = [s for s in range(sub_n) if s != test_sub]

        # Training set: all epochs from all training subjects
        # shape before reshape: (train_subjects, 40, 6, 40)
        X_train = fbcca_feats_ch[train_subs].reshape(len(train_subs) * n_trials * n_blocks, n_targets)

        # Labels: for each subject, trials 0..39 repeated for each block
        y_train_one_sub = np.repeat(np.arange(n_trials), n_blocks)  # length = 40*6
        y_train = np.tile(y_train_one_sub, len(train_subs))         # repeat for each training subject

        # Test set: all epochs from held out subject
        X_test = fbcca_feats_ch[test_sub].reshape(n_trials * n_blocks, n_targets)
        y_test = np.repeat(np.arange(n_trials), n_blocks)

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="rbf", C=1, gamma=0.01))
        ])

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        # Confusion matrix 
        cm_fold = confusion_matrix(y_test, y_pred, labels=np.arange(n_trials))
        cm_total += cm_fold

        # Accuracy for this subject fold
        acc = np.mean(y_pred == y_test)
        acc_folds.append(acc)

        # ITR for this fold
        if acc == 1:
            itr = 60 / total_t * np.log2(N)
        elif acc < 1 / N:
            itr = 0.0
        else:
            itr = 60 / total_t * (
                np.log2(N)
                + acc * np.log2(acc)
                + (1 - acc) * np.log2((1 - acc) / (N - 1))
            )
        itr_folds.append(float(itr))

    acc_mean = float(np.mean(acc_folds))
    itr_mean = float(np.mean(itr_folds))

    print("Mean accuracy:", acc_mean)
    print("Mean ITR (bits/min):", itr_mean)
    print("Confusion matrix shape:", cm_total.shape)

savemat(
    "fbcca_phase_ch_features.mat",
    {
        "features": fbcca_phase_feats_ch,        # numpy array
        "channel_labels": channel_labels,     # optional metadata
        "fs": fs,
        "epoch_idx": epoch_idx,
        "target_freqs": target_freqs,
    }
)

### Grid search

In [ ]:
tw_seq = np.arange(0.25, 3.50 + 1e-12, 0.25)

num_harms = 5
num_fbs = 7
f_high = 90.0
fbcca_preds = 3

t_cue = 0.5
t_latency = 0.14
t_break = 0.5
t_comp = 0.0

svc_params = dict(kernel="rbf", C=1, gamma=0.01)

# Setups to evaluate
setups = [
    ("All9", None),
    ("Oz",  "Oz")
]

# Output folder for this grid search
grid_dir = r"\TW_Grid"
os.makedirs(grid_dir, exist_ok=True)

target_freqs = np.asarray(labels_fr).reshape(-1)
list_freqs = np.asarray(labels_fr).reshape(-1)

def loso_eval_subjectwise(X, svc_params, tw, t_break, t_latency, t_comp):
    """
    X: (n_subjects, n_trials, n_blocks, n_features)
    Returns per-subject accuracy and ITR arrays
    """
    n_subjects, n_trials, n_blocks, n_feats = X.shape
    total_t = float(tw) + float(t_break) + float(t_latency) + float(t_comp)
    N = int(n_trials)

    y_one_sub = np.repeat(np.arange(n_trials), n_blocks)

    acc_sub = np.zeros(n_subjects, dtype=float)
    itr_sub = np.zeros(n_subjects, dtype=float)

    for test_sub in range(n_subjects):
        train_subs = [s for s in range(n_subjects) if s != test_sub]

        X_train = X[train_subs].reshape(len(train_subs) * n_trials * n_blocks, n_feats)
        y_train = np.tile(y_one_sub, len(train_subs))

        X_test = X[test_sub].reshape(n_trials * n_blocks, n_feats)
        y_test = y_one_sub

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(**svc_params)),
        ])

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        acc = float(np.mean(y_pred == y_test))
        acc_sub[test_sub] = acc

        if acc == 1.0:
            itr = 60.0 / total_t * np.log2(N)
        elif acc < 1.0 / N:
            itr = 0.0
        else:
            itr = 60.0 / total_t * (
                np.log2(N)
                + acc * np.log2(acc)
                + (1.0 - acc) * np.log2((1.0 - acc) / (N - 1))
            )
        itr_sub[test_sub] = float(itr)

    return acc_sub, itr_sub

# Compute filterbank once
sos_list = filterbank_sos(fs=fs, num_fbs=num_fbs, f_high=f_high)

# Get number of samples per trial for bounds checking
n_samples_trial = int(filtered_data[0].shape[1])

# CSV summary
rows = []

# Run grid
for setup_name, ch_label in setups:
    print("Setup:", setup_name, "| ch_label:", ch_label)

    for t_w in tw_seq:
        t_w = float(t_w)
        print("\n--- t_w =", t_w, "s ---")

        # Epoch indices and bounds check
        epoch_idx = build_epoch_length_indices(fs=fs, t_cue=t_cue, t_latency=t_latency, t_gaze=t_w)

        # Reference signals depend on epoch length
        y_ref = cca_reference(list_freqs=list_freqs, fs=fs, n_smpls=len(epoch_idx), n_harms=num_harms)

        # Feature extraction
        feats_list = []
        for sub in range(sub_n):
            feats_sub = fbcca_phase_features_per_subject(
                sub_data=filtered_data[sub],
                epoch_idx=epoch_idx,
                y_ref=y_ref,
                sos_list=sos_list,
                ch_label=ch_label,
                channel_labels=channel_labels,
                target_freqs=target_freqs,
                fbcca_preds=fbcca_preds,
                fs=fs
            )
            feats_list.append(feats_sub.astype(np.float32))

        feats = np.stack(feats_list, axis=0)  # (sub, trial, block, feat)

        # LOSO evaluation
        acc_sub, itr_sub = loso_eval_subjectwise(feats, svc_params, t_w, t_break, t_latency, t_comp)

        acc_mean = float(np.mean(acc_sub))
        acc_sd = float(np.std(acc_sub, ddof=1))
        itr_mean = float(np.mean(itr_sub))
        itr_sd = float(np.std(itr_sub, ddof=1))

        print(f"Accuracy: {acc_mean:.4f}  sd {acc_sd:.4f}")
        print(f"ITR:      {itr_mean:.4f}  sd {itr_sd:.4f} bits/min")

        # Save one MAT per variation
        tw_str = f"{t_w:.2f}".replace(".", "p")
        ch_str = "All9" if ch_label is None else ch_label
        mat_name = f"FBCCA_PHASE_{ch_str}_tw{tw_str}.mat"
        mat_path = os.path.join(grid_dir, mat_name)

        savemat(
            mat_path,
            {
                # Core data
                "features": feats,
                "acc_subject": acc_sub.astype(np.float32),
                "itr_subject": itr_sub.astype(np.float32),

                # Summary stats
                "acc_mean": np.float32(acc_mean),
                "acc_sd": np.float32(acc_sd),
                "itr_mean": np.float32(itr_mean),
                "itr_sd": np.float32(itr_sd),

                # Metadata
                "setup": setup_name,
                "ch_label": "" if ch_label is None else ch_label,
                "channel_labels": np.array(channel_labels, dtype=object),

                # Timing and indexing
                "fs": float(fs),
                "t_w": float(t_w),
                "t_cue": float(t_cue),
                "t_latency": float(t_latency),
                "t_break": float(t_break),
                "t_comp": float(t_comp),
                "epoch_idx": epoch_idx.astype(np.int32),

                # FBCCA parameters
                "target_freqs": target_freqs.astype(np.float32),
                "num_harms": int(num_harms),
                "num_fbs": int(num_fbs),
                "f_high": float(f_high),
                "fbcca_preds": int(fbcca_preds),

                # Classifier config
                "svc_params": str(svc_params),
            }
        )

        print("Saved:", mat_path)

        rows.append([setup_name, ch_str, t_w, acc_mean, acc_sd, itr_mean, itr_sd, mat_path])

# Save CSV summary
df = pd.DataFrame(
    rows,
    columns=["setup", "channel_rule", "t_w", "acc_mean", "acc_sd", "itr_mean", "itr_sd", "mat_file"]
)
csv_path = os.path.join(grid_dir, "tw_grid_results.csv")
df.to_csv(csv_path, index=False)
print("\nSaved CSV:", csv_path)

df.head()

## Classification and statistics

### Single-channel

In [ ]:
# %% Paths and constants
project_root = r"\Project"

features_old_dir = os.path.join(project_root, "Scripts", "Single Channel With Phase")
features_new_dir = os.path.join(features_old_dir, "New Phase Parameters")

out_dir = r"plots"
os.makedirs(out_dir, exist_ok=True)

fs = 250
sub_n = 35
n_trials = 40
n_blocks = 6

channel_labels = ['Pz', 'PO5', 'PO3', 'POz', 'PO4', 'PO6', 'O1', 'Oz', 'O2']

t_w = 2
t_break = 0.5
t_latency = 0.14
t_comp = 0.0
total_t = t_w + t_break + t_latency + t_comp

In [ ]:
FIG_W = 12
FIG_H = 5

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 16,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 12,
    "lines.linewidth": 1.0,
    "lines.markersize": 5.0,
    "figure.figsize": (FIG_W, FIG_H),
    "figure.dpi": 200,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
def load_first_array_from_mat(mat_path, prefer_key="features"):
    d = sio.loadmat(mat_path)
    if prefer_key in d and isinstance(d[prefer_key], np.ndarray):
        return d[prefer_key]
    for k, v in d.items():
        if k.startswith("__"):
            continue
        if isinstance(v, np.ndarray):
            return v
    raise ValueError(f"No ndarray found in {mat_path}")

def infer_channel_suffix(path):
    base = os.path.splitext(os.path.basename(path))[0]
    parts = base.split("_")
    return parts[-1]

def itr_from_acc(acc, N, total_t):
    acc = float(acc)
    if acc == 1.0:
        return float(60.0 / total_t * np.log2(N))
    if acc < 1.0 / N:
        return 0.0
    return float(
        60.0 / total_t * (
            np.log2(N)
            + acc * np.log2(acc)
            + (1 - acc) * np.log2((1 - acc) / (N - 1))
        )
    )

def star_from_p(p):
    if not np.isfinite(p):
        return ""
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    return ""

def draw_vertical_stars(ax, x, p_value, y_top, y_span, color="k"):
    if not np.isfinite(p_value):
        return
    if p_value < 1e-4:
        n = 4
    elif p_value < 1e-3:
        n = 3
    elif p_value < 1e-2:
        n = 2
    elif p_value < 5e-2:
        n = 1
    else:
        return

    dy = 0.03 * y_span
    for i in range(n):
        ax.text(
            x,
            y_top - i * dy,
            "*",
            ha="center",
            va="top",
            color=color,
            fontsize=12,
            fontweight="bold"
        )

def save_plot_data_csv(
    csv_path,
    plot_id,
    panel,
    x_values,
    series_dict,
    unit,
    p_values,
    stars,
):
    x_values = list(x_values)
    n = len(x_values)
    x_index = np.arange(n, dtype=int)

    if len(p_values) != n:
        raise ValueError("p_values must have same length as x_values")
    if len(stars) != n:
        raise ValueError("stars must have same length as x_values")

    rows = []
    for series, (y, yerr) in series_dict.items():
        y = np.asarray(y).reshape(-1)
        yerr = np.asarray(yerr).reshape(-1)
        if len(y) != n or len(yerr) != n:
            raise ValueError(f"Length mismatch in series '{series}'")

        for i in range(n):
            rows.append({
                "plot_id": plot_id,
                "panel": panel,
                "series": series,
                "x_index": int(x_index[i]),
                "x_value": x_values[i],
                "y": float(y[i]),
                "yerr": float(yerr[i]),
                "unit": unit,
                "p_value": float(p_values[i]) if np.isfinite(p_values[i]) else np.nan,
                "stars": str(stars[i]),
            })

    df = pd.DataFrame(rows)
    write_header = not os.path.exists(csv_path)
    df.to_csv(csv_path, mode="a", header=write_header, index=False)
    print(f"Saved plot data ({panel}) -> {csv_path}")
    return df


#### Classifier parameters

In [ ]:
svc_params = dict(kernel="rbf", C=1, gamma=0.01)

use_linear = False
lin_params = dict(C=1.0)

def make_clf():
    if use_linear:
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearSVC(**lin_params))
        ])
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(**svc_params))
    ])


#### Discover feature files

In [ ]:

fbcca_files = sorted(glob.glob(os.path.join(features_old_dir, "FBCCA_Feats_*")))
phase_files = sorted(glob.glob(os.path.join(features_new_dir, "Phase_Feats_*")))

fbcca_by_ch = {infer_channel_suffix(p): p for p in fbcca_files}
phase_by_ch = {infer_channel_suffix(p): p for p in phase_files}

channels_available = [ch for ch in channel_labels if (ch in fbcca_by_ch and ch in phase_by_ch)]
print("Channels available:", channels_available)


In [ ]:
def loso_eval_subjectwise_blockstats(feats, make_clf, n_trials, n_blocks, total_t):
    """
    feats: (sub, trial, block, feat)

    Returns:
      acc_mean_sub, acc_sd_sub, itr_mean_sub, itr_sd_sub, acc_blocks_sub, itr_blocks_sub
    where acc_blocks_sub and itr_blocks_sub have shape (sub, n_blocks)
    """
    feats = np.asarray(feats)
    n_subjects, nT, nB, nF = feats.shape
    if (nT, nB) != (n_trials, n_blocks):
        raise ValueError(f"Unexpected feats shape {feats.shape}")

    y_one_sub = np.repeat(np.arange(n_trials), n_blocks)

    acc_mean = np.zeros(n_subjects, dtype=float)
    acc_sd = np.zeros(n_subjects, dtype=float)
    itr_mean = np.zeros(n_subjects, dtype=float)
    itr_sd = np.zeros(n_subjects, dtype=float)

    acc_blocks_all = np.zeros((n_subjects, n_blocks), dtype=float)
    itr_blocks_all = np.zeros((n_subjects, n_blocks), dtype=float)

    for test_sub in range(n_subjects):
        train_subs = [s for s in range(n_subjects) if s != test_sub]

        X_train = feats[train_subs].reshape(len(train_subs) * n_trials * n_blocks, nF)
        y_train = np.tile(y_one_sub, len(train_subs))

        X_test = feats[test_sub].reshape(n_trials * n_blocks, nF)
        y_test = y_one_sub

        clf = make_clf()
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        y_pred_blk = y_pred.reshape(n_trials, n_blocks)
        y_true_blk = np.arange(n_trials)[:, None] * np.ones((1, n_blocks), dtype=int)

        acc_blk = np.mean(y_pred_blk == y_true_blk, axis=0)
        itr_blk = np.array([itr_from_acc(a, N=n_trials, total_t=total_t) for a in acc_blk], dtype=float)

        acc_blocks_all[test_sub, :] = acc_blk
        itr_blocks_all[test_sub, :] = itr_blk

        acc_mean[test_sub] = float(np.mean(acc_blk))
        acc_sd[test_sub] = float(np.std(acc_blk, ddof=1)) if n_blocks > 1 else 0.0
        itr_mean[test_sub] = float(np.mean(itr_blk))
        itr_sd[test_sub] = float(np.std(itr_blk, ddof=1)) if n_blocks > 1 else 0.0

    return acc_mean, acc_sd, itr_mean, itr_sd, acc_blocks_all, itr_blocks_all


#### Reading and classifying feature tensors

In [ ]:
rows = []
per_subject_rows = []

cache = {}

for ch_i, ch in enumerate(channels_available, 1):
    print(f"Evaluating channel {ch} ({ch_i}/{len(channels_available)})")

    FB = load_first_array_from_mat(fbcca_by_ch[ch]).astype(np.float32)
    PH = load_first_array_from_mat(phase_by_ch[ch]).astype(np.float32)

    acc_fb_m, acc_fb_sd, itr_fb_m, itr_fb_sd, acc_fb_blk, itr_fb_blk = loso_eval_subjectwise_blockstats(
        FB, make_clf, n_trials=n_trials, n_blocks=n_blocks, total_t=total_t
    )
    acc_ph_m, acc_ph_sd, itr_ph_m, itr_ph_sd, acc_ph_blk, itr_ph_blk = loso_eval_subjectwise_blockstats(
        PH, make_clf, n_trials=n_trials, n_blocks=n_blocks, total_t=total_t
    )

    cache[ch] = {
        "acc_fb_m": acc_fb_m, "acc_fb_sd": acc_fb_sd, "itr_fb_m": itr_fb_m, "itr_fb_sd": itr_fb_sd,
        "acc_ph_m": acc_ph_m, "acc_ph_sd": acc_ph_sd, "itr_ph_m": itr_ph_m, "itr_ph_sd": itr_ph_sd,
        "acc_fb_blk": acc_fb_blk, "itr_fb_blk": itr_fb_blk,
        "acc_ph_blk": acc_ph_blk, "itr_ph_blk": itr_ph_blk,
    }

    for s in range(sub_n):
        per_subject_rows.append({
            "channel": ch,
            "subject": s + 1,
            "acc_fbcca_mean": acc_fb_m[s],
            "acc_fbcca_sd": acc_fb_sd[s],
            "itr_fbcca_mean": itr_fb_m[s],
            "itr_fbcca_sd": itr_fb_sd[s],
            "acc_phase_mean": acc_ph_m[s],
            "acc_phase_sd": acc_ph_sd[s],
            "itr_phase_mean": itr_ph_m[s],
            "itr_phase_sd": itr_ph_sd[s],
            "acc_diff": acc_ph_m[s] - acc_fb_m[s],
            "itr_diff": itr_ph_m[s] - itr_fb_m[s],
        })

    p_acc = float(ttest_rel(acc_ph_m, acc_fb_m, nan_policy="omit").pvalue)
    p_itr = float(ttest_rel(itr_ph_m, itr_fb_m, nan_policy="omit").pvalue)

    rows.append({
        "channel": ch,
        "acc_fbcca_mean": float(np.mean(acc_fb_m)),
        "acc_fbcca_sd": float(np.std(acc_fb_m, ddof=1)),
        "acc_phase_mean": float(np.mean(acc_ph_m)),
        "acc_phase_sd": float(np.std(acc_ph_m, ddof=1)),
        "acc_pvalue": p_acc,
        "acc_sig": star_from_p(p_acc),
        "itr_fbcca_mean": float(np.mean(itr_fb_m)),
        "itr_fbcca_sd": float(np.std(itr_fb_m, ddof=1)),
        "itr_phase_mean": float(np.mean(itr_ph_m)),
        "itr_phase_sd": float(np.std(itr_ph_m, ddof=1)),
        "itr_pvalue": p_itr,
        "itr_sig": star_from_p(p_itr),
    })

summary_df = pd.DataFrame(rows).sort_values("acc_phase_mean", ascending=False).reset_index(drop=True)
per_sub_df = pd.DataFrame(per_subject_rows).sort_values(["channel", "subject"]).reset_index(drop=True)

summary_df.to_csv(os.path.join(out_dir, "channel_ranking_with_tests.csv"), index=False)
per_sub_df.to_csv(os.path.join(out_dir, "per_subject_all_channels.csv"), index=False)

print("Saved:", os.path.join(out_dir, "channel_ranking_with_tests.csv"))
print("Saved:", os.path.join(out_dir, "per_subject_all_channels.csv"))

summary_df.head()


#### Plot 1: Channel ranking with per channel p values

In [ ]:
plot_df = summary_df.copy()
x = np.arange(len(plot_df))
ch_names = plot_df["channel"].tolist()

fig, ax = plt.subplots(1, 2, figsize=(FIG_W, FIG_H), constrained_layout=True)

# Accuracy
ax0 = ax[0]
ax0.errorbar(x, plot_df["acc_fbcca_mean"].values * 100, yerr=plot_df["acc_fbcca_sd"].values * 100,
             fmt="o-", color='blue',label="FBCCA", capsize=5)
ax0.errorbar(x, plot_df["acc_phase_mean"].values * 100, yerr=plot_df["acc_phase_sd"].values * 100,
             fmt="o-",color='red', label="FBCCA + Phase", capsize=5)
ax0.set_xticks(x)
ax0.set_xticklabels(ch_names, rotation=45)
ax0.set_xlabel("Channel")
ax0.set_ylabel("Accuracy (%)")
ax0.set_ylim(-10, 130)
ax0.legend(bbox_to_anchor=(0, -0.3), loc='lower left', frameon=False)

y_span = ax0.get_ylim()[1] - ax0.get_ylim()[0]
y_top = ax0.get_ylim()[1] - 0.01 * y_span
for i in range(len(x)):
    draw_vertical_stars(ax0, i, float(plot_df["acc_pvalue"].iloc[i]), y_top, y_span)

# ITR
ax1 = ax[1]
ax1.errorbar(x, plot_df["itr_fbcca_mean"].values, yerr=plot_df["itr_fbcca_sd"].values,
             fmt="o-",color='blue', label="FBCCA", capsize=5)
ax1.errorbar(x, plot_df["itr_phase_mean"].values, yerr=plot_df["itr_phase_sd"].values,
             fmt="o-",color='red', label="FBCCA + Phase", capsize=5)
ax1.set_xticks(x)
ax1.set_xticklabels(ch_names, rotation=45)
ax1.set_xlabel("Channel")
ax1.set_ylabel("ITR (bits/min)")
ax1.set_ylim(-10, 170)
ax1.legend(bbox_to_anchor=(0, -0.3), loc='lower left', frameon=False)

y_span = ax1.get_ylim()[1] - ax1.get_ylim()[0]
y_top = ax1.get_ylim()[1] - 0.01 * y_span
for i in range(len(x)):
    draw_vertical_stars(ax1, i, float(plot_df["itr_pvalue"].iloc[i]), y_top, y_span)

# Export plot data CSV with per point p and stars
csv_path = os.path.join(out_dir, "fig_channel_ranking.csv")
if os.path.exists(csv_path):
    os.remove(csv_path)

save_plot_data_csv(
    csv_path=csv_path,
    plot_id="channel_ranking",
    panel="acc",
    x_values=ch_names,
    series_dict={
        "fbcca": (plot_df["acc_fbcca_mean"].values * 100, plot_df["acc_fbcca_sd"].values * 100),
        "phase": (plot_df["acc_phase_mean"].values * 100, plot_df["acc_phase_sd"].values * 100),
    },
    unit="percent",
    p_values=plot_df["acc_pvalue"].values,
    stars=plot_df["acc_sig"].values
)

save_plot_data_csv(
    csv_path=csv_path,
    plot_id="channel_ranking",
    panel="itr",
    x_values=ch_names,
    series_dict={
        "fbcca": (plot_df["itr_fbcca_mean"].values, plot_df["itr_fbcca_sd"].values),
        "phase": (plot_df["itr_phase_mean"].values, plot_df["itr_phase_sd"].values),
    },
    unit="bits_per_min",
    p_values=plot_df["itr_pvalue"].values,
    stars=plot_df["itr_sig"].values
)

plt.savefig(r"plots\Results_schannel.pdf")
plt.show()
print("Saved:", csv_path)


#### Plot 2: Best channel subjects, with per subject p values

In [ ]:

best_ch = summary_df.iloc[0]["channel"]
best_df = per_sub_df[per_sub_df["channel"] == best_ch].sort_values("subject").reset_index(drop=True)

acc_fb_blk = cache[best_ch]["acc_fb_blk"]
acc_ph_blk = cache[best_ch]["acc_ph_blk"]
itr_fb_blk = cache[best_ch]["itr_fb_blk"]
itr_ph_blk = cache[best_ch]["itr_ph_blk"]

# Per subject p values across blocks
p_acc_sub = np.array([ttest_rel(acc_ph_blk[s], acc_fb_blk[s], nan_policy="omit").pvalue for s in range(sub_n)], dtype=float)
p_itr_sub = np.array([ttest_rel(itr_ph_blk[s], itr_fb_blk[s], nan_policy="omit").pvalue for s in range(sub_n)], dtype=float)

stars_acc_sub = np.array([star_from_p(p) for p in p_acc_sub], dtype=object)
stars_itr_sub = np.array([star_from_p(p) for p in p_itr_sub], dtype=object)

x = best_df["subject"].values

fig, ax = plt.subplots(1, 2, figsize=(FIG_W, FIG_H), constrained_layout=True)

# Accuracy
ax0 = ax[0]
ax0.errorbar(x, best_df["acc_fbcca_mean"].values * 100, yerr=best_df["acc_fbcca_sd"].values * 100,
             fmt="o",color='blue', label="FBCCA", capsize=4)
ax0.errorbar(x, best_df["acc_phase_mean"].values * 100, yerr=best_df["acc_phase_sd"].values * 100,
             fmt="o",color='red', label="FBCCA + Phase", capsize=4)
ax0.set_xlabel("Subject")
ax0.set_ylabel("Accuracy (%)")
ax0.set_ylim(-10, 130)
ax0.legend(bbox_to_anchor=(0, -0.3), loc='lower left', frameon=False)

y_span = ax0.get_ylim()[1] - ax0.get_ylim()[0]
y_top = ax0.get_ylim()[1] - 0.01 * y_span
for i, subj in enumerate(x):
    draw_vertical_stars(ax0, subj, float(p_acc_sub[i]), y_top, y_span)

# ITR
ax1 = ax[1]
ax1.errorbar(x, best_df["itr_fbcca_mean"].values, yerr=best_df["itr_fbcca_sd"].values,
             fmt="o",color='blue', label="FBCCA", capsize=4)
ax1.errorbar(x, best_df["itr_phase_mean"].values, yerr=best_df["itr_phase_sd"].values,
             fmt="o",color='red', label="FBCCA + Phase", capsize=4)
ax1.set_xlabel("Subject")
ax1.set_ylabel("ITR (bits/min)")
ax1.set_ylim(-10, 170)
ax1.legend(bbox_to_anchor=(0, -0.3), loc='lower left', frameon=False)

y_span = ax1.get_ylim()[1] - ax1.get_ylim()[0]
y_top = ax1.get_ylim()[1] - 0.01 * y_span
for i, subj in enumerate(x):
    draw_vertical_stars(ax1, subj, float(p_itr_sub[i]), y_top, y_span)

# Export plot data CSV with per point p and stars
csv_path = os.path.join(out_dir, f"fig_best_channel_{best_ch}_subjects.csv")
if os.path.exists(csv_path):
    os.remove(csv_path)

save_plot_data_csv(
    csv_path=csv_path,
    plot_id=f"best_channel_{best_ch}_subjects",
    panel="acc",
    x_values=x,
    series_dict={
        "fbcca": (best_df["acc_fbcca_mean"].values * 100, best_df["acc_fbcca_sd"].values * 100),
        "phase": (best_df["acc_phase_mean"].values * 100, best_df["acc_phase_sd"].values * 100),
    },
    unit="percent",
    p_values=p_acc_sub,
    stars=stars_acc_sub
)

save_plot_data_csv(
    csv_path=csv_path,
    plot_id=f"best_channel_{best_ch}_subjects",
    panel="itr",
    x_values=x,
    series_dict={
        "fbcca": (best_df["itr_fbcca_mean"].values, best_df["itr_fbcca_sd"].values),
        "phase": (best_df["itr_phase_mean"].values, best_df["itr_phase_sd"].values),
    },
    unit="bits_per_min",
    p_values=p_itr_sub,
    stars=stars_itr_sub
)

plt.savefig(r"plots\Results_subject_sch.pdf")
plt.show()

print("Saved:", csv_path)


### 9-channel

In [ ]:
# %% Paths and constants
project_root = r"\Project"
features_dir = os.path.join(project_root, "Scripts", "Single Channel With Phase")

in_file = os.path.join(features_dir, "fbcca_phase_features.mat")

out_dir = os.path.join(project_root, "Scripts", "plots")
os.makedirs(out_dir, exist_ok=True)

plots_dir = os.path.join(project_root, "Scripts", "plots")
os.makedirs(plots_dir, exist_ok=True)

K = 40
svc_params = dict(kernel="rbf", C=1, gamma=0.01)

t_w = 2.0
t_break = 0.5
t_latency = 0.14
t_comp = 0.0
total_t = t_w + t_break + t_latency + t_comp

print("Input:", in_file)
print("Out:", out_dir)
print("Plots:", plots_dir)
print("Classifier:", svc_params)
print("Timing total_t:", total_t)


In [ ]:
def itr_from_acc(acc, N, total_t):
    acc = float(acc)
    if acc == 1.0:
        return float(60.0 / total_t * np.log2(N))
    if acc < 1.0 / N:
        return 0.0
    return float(
        60.0 / total_t * (
            np.log2(N)
            + acc * np.log2(acc)
            + (1 - acc) * np.log2((1 - acc) / (N - 1))
        )
    )

def star_from_p(p):
    if not np.isfinite(p):
        return ""
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    return ""

def draw_vertical_stars(ax, x, p_value, y_top, y_span, color="k"):
    if not np.isfinite(p_value):
        return
    if p_value < 1e-4:
        n = 4
    elif p_value < 1e-3:
        n = 3
    elif p_value < 1e-2:
        n = 2
    elif p_value < 5e-2:
        n = 1
    else:
        return

    dy = 0.03 * y_span
    for i in range(n):
        ax.text(
            x,
            y_top - i * dy,
            "*",
            ha="center",
            va="top",
            color=color,
            fontsize=12,
            fontweight="bold"
        )

def save_plot_data_csv(
    csv_path,
    plot_id,
    panel,
    x_values,
    series_dict,
    unit,
    p_values,
    stars,
):
    x_values = list(x_values)
    n = len(x_values)
    x_index = np.arange(n, dtype=int)

    if len(p_values) != n:
        raise ValueError("p_values must have same length as x_values")
    if len(stars) != n:
        raise ValueError("stars must have same length as x_values")

    rows = []
    for series, (y, yerr) in series_dict.items():
        y = np.asarray(y).reshape(-1)
        yerr = np.asarray(yerr).reshape(-1)
        if len(y) != n or len(yerr) != n:
            raise ValueError(f"Length mismatch in series '{series}'")

        for i in range(n):
            rows.append({
                "plot_id": plot_id,
                "panel": panel,
                "series": series,
                "x_index": int(x_index[i]),
                "x_value": x_values[i],
                "y": float(y[i]),
                "yerr": float(yerr[i]),
                "unit": unit,
                "p_value": float(p_values[i]) if np.isfinite(p_values[i]) else np.nan,
                "stars": str(stars[i]),
            })

    df = pd.DataFrame(rows)
    write_header = not os.path.exists(csv_path)
    df.to_csv(csv_path, mode="a", header=write_header, index=False)
    print(f"Saved plot data ({panel}) -> {csv_path}")
    return df


In [ ]:
def make_clf():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(**svc_params))
    ])

# Fix axis order into (sub, trial, block, feat)
def to_stbf(X):
    X = np.asarray(X)

    shape = X.shape
    candidates = []
    for s_axis in range(4):
        for t_axis in range(4):
            for b_axis in range(4):
                for f_axis in range(4):
                    if len({s_axis, t_axis, b_axis, f_axis}) != 4:
                        continue
                    s, t, b, f = shape[s_axis], shape[t_axis], shape[b_axis], shape[f_axis]
                    # basic heuristics
                    if abs(s - 35) <= 2 and t == 40 and b == 6 and f >= 40:
                        candidates.append((s_axis, t_axis, b_axis, f_axis))

    if not candidates:
        # fallback: assume already correct
        return X

    # take first match
    s_axis, t_axis, b_axis, f_axis = candidates[0]
    X2 = np.moveaxis(X, [s_axis, t_axis, b_axis, f_axis], [0, 1, 2, 3])
    return X2

#### Load MAT and locate feature array

In [ ]:
mat = sio.loadmat(in_file)
keys = [k for k in mat.keys() if not k.startswith("__")]
print("MAT keys:", keys)

feat_key = "features" if "features" in keys else keys[0]
X_raw = np.asarray(mat[feat_key])

print("Loaded key:", feat_key)
print("Raw shape:", X_raw.shape, "dtype:", X_raw.dtype)

X_all = to_stbf(X_raw).astype(np.float32)
print("Coerced shape (sub, trial, block, feat):", X_all.shape)

n_subjects, n_trials, n_blocks, n_features = X_all.shape
print("Subjects:", n_subjects, "Trials:", n_trials, "Blocks:", n_blocks, "Features:", n_features)

In [ ]:
# LOSO returning per subject per block accuracy and ITR
def loso_eval_subject_blockwise(X, make_clf):
    X = np.asarray(X)
    nS, nT, nB, nF = X.shape

    y_one = np.repeat(np.arange(nT), nB)

    acc_blk = np.zeros((nS, nB), dtype=float)
    itr_blk = np.zeros((nS, nB), dtype=float)

    acc_mean = np.zeros(nS, dtype=float)
    acc_sd = np.zeros(nS, dtype=float)
    itr_mean = np.zeros(nS, dtype=float)
    itr_sd = np.zeros(nS, dtype=float)

    for s in range(nS):
        train = [i for i in range(nS) if i != s]

        Xtr = X[train].reshape(len(train) * nT * nB, nF)
        ytr = np.tile(y_one, len(train))

        Xte = X[s].reshape(nT * nB, nF)
        yte = y_one

        clf = make_clf()
        clf.fit(Xtr, ytr)
        yhat = clf.predict(Xte)

        for bl in range(nB):
            sl = slice(bl * nT, (bl + 1) * nT)
            a = float(np.mean(yhat[sl] == yte[sl]))
            acc_blk[s, bl] = a
            itr_blk[s, bl] = itr_from_acc(a, N=nT, total_t=total_t)

        acc_mean[s] = float(np.mean(acc_blk[s]))
        acc_sd[s] = float(np.std(acc_blk[s], ddof=1)) if nB > 1 else 0.0
        itr_mean[s] = float(np.mean(itr_blk[s]))
        itr_sd[s] = float(np.std(itr_blk[s], ddof=1)) if nB > 1 else 0.0

        if (s + 1) % 5 == 0 or (s + 1) == nS:
            print(f"Progress: subject {s + 1}/{nS}")

    return acc_blk, itr_blk, acc_mean, acc_sd, itr_mean, itr_sd


In [ ]:
X_fbcca = X_all[..., :K]
X_phase = X_all

print("X_fbcca:", X_fbcca.shape)
print("X_phase:", X_phase.shape)
print("Phase tail length:", n_features - K)

print("Running LOSO: FBCCA")
acc_fb_blk, itr_fb_blk, acc_fb_m, acc_fb_sd, itr_fb_m, itr_fb_sd = loso_eval_subject_blockwise(X_fbcca, make_clf)

print("Running LOSO: FBCCA + Phase")
acc_ph_blk, itr_ph_blk, acc_ph_m, acc_ph_sd, itr_ph_m, itr_ph_sd = loso_eval_subject_blockwise(X_phase, make_clf)

print("Done rerun")

In [ ]:
# Per subject per point t tests across blocks
p_acc_sub = np.array([ttest_rel(acc_ph_blk[s], acc_fb_blk[s], nan_policy="omit").pvalue for s in range(n_subjects)], dtype=float)
p_itr_sub = np.array([ttest_rel(itr_ph_blk[s], itr_fb_blk[s], nan_policy="omit").pvalue for s in range(n_subjects)], dtype=float)

stars_acc_sub = np.array([star_from_p(p) for p in p_acc_sub], dtype=object)
stars_itr_sub = np.array([star_from_p(p) for p in p_itr_sub], dtype=object)

print("Acc p value range:", np.nanmin(p_acc_sub), "to", np.nanmax(p_acc_sub))
print("ITR p value range:", np.nanmin(p_itr_sub), "to", np.nanmax(p_itr_sub))

In [ ]:
# Save per subject results table
df_sub = pd.DataFrame({
    "subject": np.arange(1, n_subjects + 1, dtype=int),

    "acc_fbcca_mean": acc_fb_m,
    "acc_fbcca_sd": acc_fb_sd,
    "itr_fbcca_mean": itr_fb_m,
    "itr_fbcca_sd": itr_fb_sd,

    "acc_phase_mean": acc_ph_m,
    "acc_phase_sd": acc_ph_sd,
    "itr_phase_mean": itr_ph_m,
    "itr_phase_sd": itr_ph_sd,

    "acc_pvalue": p_acc_sub,
    "acc_stars": stars_acc_sub,
    "itr_pvalue": p_itr_sub,
    "itr_stars": stars_itr_sub,
})

csv_sub = os.path.join(out_dir, "per_subject_9ch_fbcca_vs_phase.csv")
df_sub.to_csv(csv_sub, index=False)
print("Saved:", csv_sub)


In [ ]:
# Export plot CSV 
plot_csv = os.path.join(out_dir, "fig_9ch_subjects.csv")
if os.path.exists(plot_csv):
    os.remove(plot_csv)

subjects = np.arange(1, n_subjects + 1, dtype=int)

save_plot_data_csv(
    csv_path=plot_csv,
    plot_id="9ch_subjects",
    panel="acc",
    x_values=subjects,
    series_dict={
        "fbcca": (acc_fb_m * 100, acc_fb_sd * 100),
        "phase": (acc_ph_m * 100, acc_ph_sd * 100),
    },
    unit="percent",
    p_values=p_acc_sub,
    stars=stars_acc_sub
)

save_plot_data_csv(
    csv_path=plot_csv,
    plot_id="9ch_subjects",
    panel="itr",
    x_values=subjects,
    series_dict={
        "fbcca": (itr_fb_m, itr_fb_sd),
        "phase": (itr_ph_m, itr_ph_sd),
    },
    unit="bits_per_min",
    p_values=p_itr_sub,
    stars=stars_itr_sub
)

print("Saved:", plot_csv)


#### Plot: subjects 

In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(FIG_W, FIG_H), constrained_layout=True)

ax0 = ax[0]
ax0.errorbar(subjects, acc_fb_m * 100, yerr=acc_fb_sd * 100, fmt="o",color='blue', capsize=4, label="FBCCA")
ax0.errorbar(subjects, acc_ph_m * 100, yerr=acc_ph_sd * 100, fmt="o",color='red', capsize=4, label="FBCCA + Phase")
ax0.set_xlabel("Subject")
ax0.set_ylabel("Accuracy (%)")
ax0.set_ylim(-10, 130)
ax0.legend(bbox_to_anchor=(0, -0.3), loc='lower left', frameon=False)

y_span = ax0.get_ylim()[1] - ax0.get_ylim()[0]
y_top = ax0.get_ylim()[1] - 0.01 * y_span
for i, s in enumerate(subjects):
    draw_vertical_stars(ax0, s, float(p_acc_sub[i]), y_top, y_span)

ax1 = ax[1]
ax1.errorbar(subjects, itr_fb_m, yerr=itr_fb_sd, fmt="o",color='blue', capsize=4, label="FBCCA")
ax1.errorbar(subjects, itr_ph_m, yerr=itr_ph_sd, fmt="o",color='red', capsize=4, label="FBCCA + Phase")
ax1.set_xlabel("Subject")
ax1.set_ylabel("ITR (bits/min)")
ax1.set_ylim(-10, 170)
ax1.legend(bbox_to_anchor=(0, -0.3), loc='lower left', frameon=False)

y_span = ax1.get_ylim()[1] - ax1.get_ylim()[0]
y_top = ax1.get_ylim()[1] - 0.01 * y_span
for i, s in enumerate(subjects):
    draw_vertical_stars(ax1, s, float(p_itr_sub[i]), y_top, y_span)

pdf_out = os.path.join(plots_dir, "Results_subjects_9_channels.pdf")
plt.savefig(pdf_out)
plt.show()
print("Saved:", pdf_out)


## Time Window Grid Analysis

In [ ]:
grid_dir = r"\TW_Grid"
out_dir = grid_dir

svc_params = dict(kernel="rbf", C=1, gamma=0.01)
K = 40
setups = ["All9", "Oz"]


# Helpers
def _parse_tw(fname):
    base = os.path.splitext(os.path.basename(fname))[0]
    m = re.search(r"_tw(?P<tw>\d+p\d+)$", base)
    if not m:
        return None
    return float(m.group("tw").replace("p", "."))

def _parse_setup_from_fname(fname):
    base = os.path.splitext(os.path.basename(fname))[0]
    m = re.search(r"FBCCA_PHASE_(?P<setup>.+?)_tw\d+p\d+$", base)
    if not m:
        return None
    return m.group("setup")

def loso_eval_subjectwise(X, svc_params, t_w, t_break, t_latency, t_comp):
    n_subjects, n_trials_local, n_blocks_local, n_feats = X.shape
    if (n_trials_local != n_trials) or (n_blocks_local != n_blocks):
        raise ValueError(f"Unexpected shape {X.shape}")

    total_t = float(t_w) + float(t_break) + float(t_latency) + float(t_comp)
    y_one = np.repeat(np.arange(n_trials), n_blocks)

    acc = np.zeros(n_subjects, dtype=float)
    itr = np.zeros(n_subjects, dtype=float)

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(**svc_params)),
    ])

    for s in range(n_subjects):
        train = [i for i in range(n_subjects) if i != s]

        Xtr = X[train].reshape(len(train) * n_trials * n_blocks, n_feats)
        ytr = np.tile(y_one, len(train))

        Xte = X[s].reshape(n_trials * n_blocks, n_feats)
        yte = y_one

        pipe.fit(Xtr, ytr)
        yhat = pipe.predict(Xte)

        a = float(np.mean(yhat == yte))
        acc[s] = a
        itr[s] = itr_from_acc(a, N=n_trials, total_t=total_t)

    return acc, itr

# Discover files
all_files = sorted(glob.glob(os.path.join(grid_dir, "FBCCA_PHASE_*_tw*.mat")))
if not all_files:
    raise FileNotFoundError(f"No grid MAT files found in {grid_dir}")

print(f"Found {len(all_files)} grid files")
print("Setups to analyze:", setups)

# Run analysis per setup
t_global_start = time.time()

for setup in setups:
    print("Starting setup:", setup)

    setup_files = [f for f in all_files if _parse_setup_from_fname(f) == setup]
    if not setup_files:
        print("No files found for setup:", setup)
        continue

    setup_files = sorted(setup_files, key=_parse_tw)
    print(f"{len(setup_files)} time windows detected")

    rows = []
    long_rows = []

    t_setup_start = time.time()

    for i, mp in enumerate(setup_files, 1):
        t_w = _parse_tw(mp)
        if t_w is None:
            continue

        print(f"\n[{setup}] t_w = {t_w:.2f} s  ({i}/{len(setup_files)})")

        d = sio.loadmat(mp)
        X = np.asarray(d["features"]).astype(np.float32)

        t_break = float(np.asarray(d["t_break"]).squeeze())
        t_latency = float(np.asarray(d["t_latency"]).squeeze())
        t_comp = float(np.asarray(d["t_comp"]).squeeze())

        # Split features into FBCCA and FBCCA + Phase
        X_fb = X[..., :K]
        X_ph = X

        print("  Running LOSO: FBCCA")
        acc_fb_sub, itr_fb_sub = loso_eval_subjectwise(
            X_fb, svc_params, t_w, t_break, t_latency, t_comp
        )

        print("  Running LOSO: FBCCA + Phase")
        acc_ph_sub, itr_ph_sub = loso_eval_subjectwise(
            X_ph, svc_params, t_w, t_break, t_latency, t_comp
        )

        # Paired tests
        p_acc = float(ttest_rel(acc_ph_sub, acc_fb_sub, nan_policy="omit").pvalue)
        p_itr = float(ttest_rel(itr_ph_sub, itr_fb_sub, nan_policy="omit").pvalue)

        print(
            f"  Acc: FBCCA {acc_fb_sub.mean():.3f}, Phase {acc_ph_sub.mean():.3f}, p={p_acc:.3g}"
        )
        print(
            f"  ITR: FBCCA {itr_fb_sub.mean():.2f}, Phase {itr_ph_sub.mean():.2f}, p={p_itr:.3g}"
        )

        rows.append({
            "setup": setup,
            "t_w": float(t_w),
            "t_break": t_break,
            "t_latency": t_latency,
            "t_comp": t_comp,

            "acc_fb_mean": float(np.mean(acc_fb_sub)),
            "acc_fb_sd": float(np.std(acc_fb_sub, ddof=1)),
            "itr_fb_mean": float(np.mean(itr_fb_sub)),
            "itr_fb_sd": float(np.std(itr_fb_sub, ddof=1)),

            "acc_ph_mean": float(np.mean(acc_ph_sub)),
            "acc_ph_sd": float(np.std(acc_ph_sub, ddof=1)),
            "itr_ph_mean": float(np.mean(itr_ph_sub)),
            "itr_ph_sd": float(np.std(itr_ph_sub, ddof=1)),

            "acc_pvalue": p_acc,
            "itr_pvalue": p_itr,

            "mat_file": mp,
        })

        for s in range(len(acc_fb_sub)):
            long_rows.append({
                "setup": setup,
                "t_w": float(t_w),
                "subject": int(s + 1),
                "acc_fb": float(acc_fb_sub[s]),
                "itr_fb": float(itr_fb_sub[s]),
                "acc_ph": float(acc_ph_sub[s]),
                "itr_ph": float(itr_ph_sub[s]),
                "acc_diff": float(acc_ph_sub[s] - acc_fb_sub[s]),
                "itr_diff": float(itr_ph_sub[s] - itr_fb_sub[s]),
            })

    # Save outputs for this setup
    res_df = pd.DataFrame(rows).sort_values("t_w").reset_index(drop=True)
    long_df = pd.DataFrame(long_rows).sort_values(["t_w", "subject"]).reset_index(drop=True)

    res_csv = os.path.join(out_dir, f"tw_grid_fbcca_vs_phase_{setup}.csv")
    long_csv = os.path.join(out_dir, f"tw_grid_fbcca_vs_phase_{setup}_per_subject.csv")

    res_df.to_csv(res_csv, index=False)
    long_df.to_csv(long_csv, index=False)

    t_setup_end = time.time()
    print(f"\nFinished setup {setup} in {(t_setup_end - t_setup_start)/60:.1f} minutes")
    print("Saved:", res_csv)
    print("Saved:", long_csv)

t_global_end = time.time()
print(f"All setups finished in {(t_global_end - t_global_start)/60:.1f} minutes")


In [ ]:
# Plotting TW gridsearches
out_dir = r"\TW_Grid"
setups = ["All9", "Oz"]

for setup in setups:
    csv_in = os.path.join(out_dir, f"tw_grid_fbcca_vs_phase_{setup}.csv")
    if not os.path.exists(csv_in):
        print("Missing:", csv_in)
        continue

    df = pd.read_csv(csv_in).sort_values("t_w").reset_index(drop=True)

    tw = df["t_w"].values

    # accuracy in percent for plotting
    acc_fb_m = df["acc_fb_mean"].values * 100
    acc_fb_sd = df["acc_fb_sd"].values * 100
    acc_ph_m = df["acc_ph_mean"].values * 100
    acc_ph_sd = df["acc_ph_sd"].values * 100

    itr_fb_m = df["itr_fb_mean"].values
    itr_fb_sd = df["itr_fb_sd"].values
    itr_ph_m = df["itr_ph_mean"].values
    itr_ph_sd = df["itr_ph_sd"].values

    acc_p = df["acc_pvalue"].values
    itr_p = df["itr_pvalue"].values

    acc_stars = [star_from_p(float(p)) for p in acc_p]
    itr_stars = [star_from_p(float(p)) for p in itr_p]

    fig, ax = plt.subplots(1, 2, figsize=(FIG_W, FIG_H), constrained_layout=True)

    # Accuracy
    ax0 = ax[0]
    ax0.errorbar(tw, acc_fb_m, yerr=acc_fb_sd, fmt="o",color='blue',capsize=4, label="FBCCA")
    ax0.errorbar(tw, acc_ph_m, yerr=acc_ph_sd, fmt="o",color='red',capsize=4, label="FBCCA + Phase")
    ax0.set_xlabel("Time (s)")
    ax0.set_ylabel("Accuracy (%)")
    ax0.set_ylim(-10, 130)
    ax0.legend(bbox_to_anchor=(0, -0.3), loc='lower left', frameon=False)

    y_span = ax0.get_ylim()[1] - ax0.get_ylim()[0]
    y_top = ax0.get_ylim()[1] - 0.01 * y_span
    for i, tw_i in enumerate(tw):
        if acc_stars[i]:
            draw_vertical_stars(ax0, tw_i, float(acc_p[i]), y_top, y_span)

    # ITR
    ax1 = ax[1]
    ax1.errorbar(tw, itr_fb_m, yerr=itr_fb_sd, fmt="o",color='blue',capsize=4, label="FBCCA")
    ax1.errorbar(tw, itr_ph_m, yerr=itr_ph_sd, fmt="o",color='red',capsize=4, label="FBCCA + Phase")
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("ITR (bits/min)")
    ax1.set_ylim(-10, 170)
    ax1.legend(bbox_to_anchor=(0, -0.3), loc='lower left', frameon=False)

    y_span = ax1.get_ylim()[1] - ax1.get_ylim()[0]
    y_top = ax1.get_ylim()[1] - 0.01 * y_span
    for i, tw_i in enumerate(tw):
        if itr_stars[i]:
            draw_vertical_stars(ax1, tw_i, float(itr_p[i]), y_top, y_span)

    # Save figure
    pdf_out = os.path.join(out_dir, f"Results_tw_grid_{setup}.pdf")
    plt.savefig(pdf_out)
    plt.show()
    print("Saved:", pdf_out)

    # Save plot data used
    plot_csv = os.path.join(out_dir, f"fig_tw_grid_{setup}.csv")

    save_plot_data_csv(
        csv_path=plot_csv,
        plot_id=f"tw_grid_{setup}",
        panel="acc",
        x_values=[f"{v:.2f}" for v in tw],
        series_dict={
            "fbcca": (acc_fb_m, acc_fb_sd),
            "phase": (acc_ph_m, acc_ph_sd),
        },
        unit="percent",
        p_values=acc_p,
        stars=acc_stars,
    )

    save_plot_data_csv(
        csv_path=plot_csv,
        plot_id=f"tw_grid_{setup}",
        panel="itr",
        x_values=[f"{v:.2f}" for v in tw],
        series_dict={
            "fbcca": (itr_fb_m, itr_fb_sd),
            "phase": (itr_ph_m, itr_ph_sd),
        },
        unit="bits_per_min",
        p_values=itr_p,
        stars=itr_stars,
    )
